# Matcha-TTS: A fast TTS architecture with conditional flow matching
---
[Shivam Mehta](https://www.kth.se/profile/smehta), [Ruibo Tu](https://www.kth.se/profile/ruibo), [Jonas Beskow](https://www.kth.se/profile/beskow), [Éva Székely](https://www.kth.se/profile/szekely), and [Gustav Eje Henter](https://people.kth.se/~ghe/)

We introduce Matcha-TTS, a new encoder-decoder architecture for speedy TTS acoustic modelling, trained using optimal-transport conditional flow matching (OT-CFM). This yields an ODE-based decoder capable of high output quality in fewer synthesis steps than models trained using score matching. Careful design choices additionally ensure each synthesis step is fast to run. The method is probabilistic, non-autoregressive, and learns to speak from scratch without external alignments. Compared to strong pre-trained baseline models, the Matcha-TTS system has the smallest memory footprint, rivals the speed of the fastest models on long utterances, and attains the highest mean opinion score in a listening test.

Demo Page: https://shivammehta25.github.io/Matcha-TTS \
Code: https://github.com/shivammehta25/Matcha-TTS




In [1]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [2]:
import datetime as dt
from pathlib import Path

import IPython.display as ipd
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm

# Hifigan imports
from matcha.hifigan.config import v1
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.env import AttrDict
from matcha.hifigan.models import Generator as HiFiGAN
# Matcha imports
from matcha.models.matcha_tts import MatchaTTS
from matcha.text import sequence_to_text, text_to_sequence
from matcha.utils.model import denormalize
from matcha.utils.utils import get_user_data_dir, intersperse
import pytorch_lightning as pl

pl.seed_everything(1234)

Seed set to 1234


1234

In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
# This allows for real time code changes being reflected in the notebook, no need to restart the kernel

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Filepaths

In [5]:
get_user_data_dir()

PosixPath('/home/sandra/.local/share/matcha_tts')

In [ ]:
MATCHA_CHECKPOINT = "/path/eva_exp1_ms/runs/2025-07-15_16-30-06/checkpoints/last.ckpt"
HIFIGAN_CHECKPOINT = get_user_data_dir() / "hifigan_univ_v1"
OUTPUT_FOLDER = "synth_output"

## Load Matcha-TTS

In [9]:
def load_model(checkpoint_path):
    model = MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)
    model.eval()
    return model
count_params = lambda x: f"{sum(p.numel() for p in x.parameters()):,}"


model = load_model(MATCHA_CHECKPOINT)
print(f"Model loaded! Parameter count: {count_params(model)}")

Model loaded! Parameter count: 20,853,985


/home/sandra/miniconda3/envs/matcha-tts/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


## Load HiFi-GAN (Vocoder)

In [10]:
def load_vocoder(checkpoint_path):
    h = AttrDict(v1)
    hifigan = HiFiGAN(h).to(device)
    hifigan.load_state_dict(torch.load(checkpoint_path, map_location=device)['generator'])
    _ = hifigan.eval()
    hifigan.remove_weight_norm()
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
denoiser = Denoiser(vocoder, mode='zeros')

/home/sandra/miniconda3/envs/matcha-tts/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...


### Helper functions to synthesise

In [11]:
@torch.inference_mode()
def process_text(text: str):
    x = torch.tensor(intersperse(text_to_sequence(text, ['estonian_cleaners_plain_text'])[0], 0),dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]],dtype=torch.long, device=device)
    x_phones = sequence_to_text(x.squeeze(0).tolist())
    return {
        'x_orig': text,
        'x': x,
        'x_lengths': x_lengths,
        'x_phones': x_phones
    }


@torch.inference_mode()
def synthesise(text, spks=None):
    text_processed = process_text(text)
    start_t = dt.datetime.now()
    output = model.synthesise(
        text_processed['x'], 
        text_processed['x_lengths'],
        n_timesteps=n_timesteps,
        temperature=temperature,
        spks=spks,
        length_scale=length_scale
    )
    # merge everything to one dict    
    output.update({'start_t': start_t, **text_processed})
    return output

@torch.inference_mode()
def to_waveform(mel, vocoder):
    audio = vocoder(mel).clamp(-1, 1)
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    return audio.cpu().squeeze()
    
def save_to_folder(filename: str, output: dict, folder: str):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    np.save(folder / f'{filename}', output['mel'].cpu().numpy())
    sf.write(folder / f'{filename}.wav', output['waveform'], 22050, 'PCM_24')

## Setup text to synthesise

In [12]:
texts = [
    "k`ongr`es's",
    "kongress",
    "pl`aan' oli det`ailideni läbi m`õel=dud",
    "plaan oli detailideni läbi mõeldud",
    "olen k`uulnud `et sel `aastal tuleb k`ülm t`al'v",
    "olen kuulnud et sel aastal tuleb külm talv",
]

### Hyperparameters

In [13]:
## Number of ODE Solver steps
n_timesteps = 10

## Changes to the speaking rate
length_scale=1.0

## Sampling temperature
temperature = 0.667

## Synthesis

In [ ]:
outputs, rtfs = [], []
rtfs_w = []
for i, text in enumerate(tqdm(texts)):
    output = synthesise(text, spks=torch.tensor([0], device=device, dtype=torch.long))
    # For the second speaker in the pretrained model, use:
    # output = synthesise(text, spks=torch.tensor([1], device=device, dtype=torch.long))
    output['waveform'] = to_waveform(output['mel'], vocoder)

    # Compute Real Time Factor (RTF) with HiFi-GAN
    t = (dt.datetime.now() - output['start_t']).total_seconds()
    rtf_w = t * 22050 / (output['waveform'].shape[-1])

    ## Pretty print
    print(f"{'*' * 53}")
    print(f"Input text - {i}")
    print(f"{'-' * 53}")
    print(output['x_orig'])
    print(f"{'*' * 53}")
    print(f"Phonetised text - {i}")
    print(f"{'-' * 53}")
    print(output['x_phones'])
    print(f"{'*' * 53}")
    print(f"RTF:\t\t{output['rtf']:.6f}")
    print(f"RTF Waveform:\t{rtf_w:.6f}")
    rtfs.append(output['rtf'])
    rtfs_w.append(rtf_w)

    ## Display the synthesised waveform
    ipd.display(ipd.Audio(output['waveform'], rate=22050))

    ## Save the generated waveform
    save_to_folder(i, output, OUTPUT_FOLDER)

print(f"Number of ODE steps: {n_timesteps}")
print(f"Mean RTF:\t\t\t\t{np.mean(rtfs):.6f} ± {np.std(rtfs):.6f}")
print(f"Mean RTF Waveform (incl. vocoder):\t{np.mean(rtfs_w):.6f} ± {np.std(rtfs_w):.6f}")

  0%|          | 0/6 [00:00<?, ?it/s]

*****************************************************
Input text - 0
-----------------------------------------------------
k`ongr`es's
*****************************************************
Phonetised text - 0
-----------------------------------------------------
_k_`_o_n_g_r_`_e_s_'_s_
*****************************************************
RTF:		0.053961
RTF Waveform:	0.065028


*****************************************************
Input text - 1
-----------------------------------------------------
kongress
*****************************************************
Phonetised text - 1
-----------------------------------------------------
_k_o_n_g_r_e_s_s_
*****************************************************
RTF:		0.058794
RTF Waveform:	0.070173


*****************************************************
Input text - 2
-----------------------------------------------------
pl`aan' oli det`ailideni läbi m`õel=dud
*****************************************************
Phonetised text - 2
-----------------------------------------------------
_p_l_`_a_a_n_'_ _o_l_i_ _d_e_t_`_a_i_l_i_d_e_n_i_ _l_ä_b_i_ _m_`_õ_e_l_=_d_u_d_
*****************************************************
RTF:		0.022642
RTF Waveform:	0.032785


*****************************************************
Input text - 3
-----------------------------------------------------
plaan oli detailideni läbi mõeldud
*****************************************************
Phonetised text - 3
-----------------------------------------------------
_p_l_a_a_n_ _o_l_i_ _d_e_t_a_i_l_i_d_e_n_i_ _l_ä_b_i_ _m_õ_e_l_d_u_d_
*****************************************************
RTF:		0.024883
RTF Waveform:	0.034697


*****************************************************
Input text - 4
-----------------------------------------------------
olen k`uulnud `et sel `aastal tuleb k`ülm t`al'v
*****************************************************
Phonetised text - 4
-----------------------------------------------------
_o_l_e_n_ _k_`_u_u_l_n_u_d_ _`_e_t_ _s_e_l_ _`_a_a_s_t_a_l_ _t_u_l_e_b_ _k_`_ü_l_m_ _t_`_a_l_'_v_
*****************************************************
RTF:		0.021686
RTF Waveform:	0.031274


*****************************************************
Input text - 5
-----------------------------------------------------
olen kuulnud et sel aastal tuleb külm talv
*****************************************************
Phonetised text - 5
-----------------------------------------------------
_o_l_e_n_ _k_u_u_l_n_u_d_ _e_t_ _s_e_l_ _a_a_s_t_a_l_ _t_u_l_e_b_ _k_ü_l_m_ _t_a_l_v_
*****************************************************
RTF:		0.022217
RTF Waveform:	0.031961


Number of ODE steps: 10
Mean RTF:				0.034030 ± 0.015895
Mean RTF Waveform (incl. vocoder):	0.044319 ± 0.016562
